In [ ]:
!pip install pandas scikit-learn nltk flask # or fastapi uvicorn
# If using nltk, you might need to download data:
# python -m nltk.downloader punkt stopwords wordnet

In [ ]:
import pandas as pd
import os

# --- Configuration ---
# KEEP the filename as is for now, even if it's not actually gzipped
DATA_FILE = 'movie_data.csv'
# --- End Configuration ---

df = None # Initialize df to None

# Check if the file exists
if not os.path.exists(DATA_FILE):
    print(f"Error: Data file '{DATA_FILE}' not found.")
    print("Please double-check the file exists in the same directory as the script.")
    print("Or try downloading it again from: https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch08/movie_data.csv.gz")
    exit() # Stop execution if file is missing

# Try loading as a regular CSV first (most likely scenario)
try:
    print(f"Attempting to load '{DATA_FILE}' as a regular CSV...")
    df = pd.read_csv('movie_data.csv') # REMOVED compression='gzip'
    print(f"Successfully loaded {len(df)} reviews (as regular CSV).")
except Exception as e_csv:
    print(f"Failed to load as regular CSV: {e_csv}")
    # If that fails, try loading as gzipped CSV
    try:
        print(f"\nAttempting to load '{DATA_FILE}' as a gzipped CSV...")
        df = pd.read_csv(DATA_FILE, compression='gzip')
        print(f"Successfully loaded {len(df)} reviews (as gzipped CSV).")
    except Exception as e_gzip:
        print(f"Failed to load as gzipped CSV: {e_gzip}")
        print("\nError: Could not load the data file either as regular or gzipped CSV.")
        print("Please check the file format or try downloading it again.")
        exit() # Stop execution if both fail

# Now we should definitely have df if we haven't exited
if df is not None:
    # Display the first few rows to understand the structure
    print("\nFirst 5 rows of the dataset:")
    print(df.head())

    # Display information about the columns and data types
    print("\nDataset Info:")
    df.info()

    # Check the distribution of sentiments (0 = negative, 1 = positive)
    print("\nSentiment Distribution:")
    print(df['sentiment'].value_counts())
else:
    print("Error: DataFrame 'df' was not loaded.")

Attempting to load 'movie_data.csv' as a regular CSV...
Successfully loaded 50000 reviews (as regular CSV).

First 5 rows of the dataset:
                                              review  sentiment
0  In 1974, the teenager Martha Moxley (Maggie Gr...          1
1  OK... so... I really like Kris Kristofferson a...          0
2  ***SPOILER*** Do not read this, if you think a...          0
3  hi for all the people who have seen this wonde...          1
4  I recently bought the DVD, forgetting just how...          0

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 781.4+ KB

Sentiment Distribution:
sentiment
1    25000
0    25000
Name: count, dtype: int64


In [ ]:
import pandas as pd
import os
import re # Import the regular expression module

# --- Configuration ---
DATA_FILE = 'movie_data.csv' # Keep using the same filename
# --- End Configuration ---

df = None # Initialize df to None

# Check if the file exists
if not os.path.exists(DATA_FILE):
    print(f"Error: Data file '{DATA_FILE}' not found.")
    exit()

# Try loading as a regular CSV first
try:
    # print(f"Attempting to load '{DATA_FILE}' as a regular CSV...") # You can comment these prints out now
    df = pd.read_csv(DATA_FILE)
    print(f"Successfully loaded {len(df)} reviews.")
except Exception as e_csv:
    # print(f"Failed to load as regular CSV: {e_csv}")
    # If that fails, try loading as gzipped CSV
    try:
        # print(f"\nAttempting to load '{DATA_FILE}' as a gzipped CSV...")
        df = pd.read_csv(DATA_FILE, compression='gzip')
        print(f"Successfully loaded {len(df)} reviews (as gzipped CSV).")
    except Exception as e_gzip:
        print(f"Failed to load as gzipped CSV: {e_gzip}")
        print("\nError: Could not load the data file.")
        exit()

# --- Step 2: Preprocessing ---

def preprocess_text(text):
    """Cleans the input text."""
    # Convert to lowercase
    text = text.lower()
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove punctuation and numbers (keep only letters and spaces)
    text = re.sub(r'[^a-z\s]', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply the preprocessing function to the 'review' column
# This might take a moment for 50,000 reviews
print("\nPreprocessing text data...")
df['cleaned_review'] = df['review'].apply(preprocess_text)
print("Preprocessing complete.")

# Display original vs. cleaned review for a sample
print("\nSample Preprocessing Result:")
sample_index = 0 # Look at the first review
print("Original Review:")
print(df['review'].iloc[sample_index])
print("\nCleaned Review:")
print(df['cleaned_review'].iloc[sample_index])

# --- End Step 2 ---

# Display first 5 rows with the new cleaned column
print("\nDataFrame with Cleaned Reviews:")
print(df.head())

# (We'll add feature extraction and model training next)

Successfully loaded 50000 reviews.

Preprocessing text data...
Preprocessing complete.

Sample Preprocessing Result:
Original Review:
In 1974, the teenager Martha Moxley (Maggie Grace) moves to the high-class area of Belle Haven, Greenwich, Connecticut. On the Mischief Night, eve of Halloween, she was murdered in the backyard of her house and her murder remained unsolved. Twenty-two years later, the writer Mark Fuhrman (Christopher Meloni), who is a former LA detective that has fallen in disgrace for perjury in O.J. Simpson trial and moved to Idaho, decides to investigate the case with his partner Stephen Weeks (Andrew Mitchell) with the purpose of writing a book. The locals squirm and do not welcome them, but with the support of the retired detective Steve Carroll (Robert Forster) that was in charge of the investigation in the 70's, they discover the criminal and a net of power and money to cover the murder.<br /><br />"Murder in Greenwich" is a good TV movie, with the true story of a

In [ ]:
import pandas as pd
import os
import re
from sklearn.model_selection import train_test_split # Import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer # Import TfidfVectorizer

# --- Configuration ---
DATA_FILE = 'movie_data.csv'
TEST_SET_SIZE = 0.2 # Use 20% of data for testing
RANDOM_STATE = 42 # For reproducible splits
TFIDF_MAX_FEATURES = 5000 # Limit vocabulary size
# --- End Configuration ---

df = None

# ... (Keep the data loading code from the previous step) ...
# Check if the file exists
if not os.path.exists(DATA_FILE):
    print(f"Error: Data file '{DATA_FILE}' not found.")
    exit()
# Load data (try CSV, then Gzipped CSV)
try:
    df = pd.read_csv(DATA_FILE)
    print(f"Successfully loaded {len(df)} reviews.")
except Exception as e_csv:
    try:
        df = pd.read_csv(DATA_FILE, compression='gzip')
        print(f"Successfully loaded {len(df)} reviews (as gzipped CSV).")
    except Exception as e_gzip:
        print(f"Failed to load data: {e_gzip}")
        exit()

# --- Step 2: Preprocessing ---
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("\nPreprocessing text data...")
df['cleaned_review'] = df['review'].apply(preprocess_text)
print("Preprocessing complete.")
# print("\nSample Preprocessing Result:") # Can comment out if needed
# sample_index = 0
# print("Original Review:", df['review'].iloc[sample_index])
# print("\nCleaned Review:", df['cleaned_review'].iloc[sample_index])
# --- End Step 2 ---


# --- Step 3: Feature Extraction ---

# Define features (X) and target (y)
X = df['cleaned_review']
y = df['sentiment']

# Split data into training and testing sets
print(f"\nSplitting data into training ({1-TEST_SET_SIZE:.0%}) and testing ({TEST_SET_SIZE:.0%}) sets...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SET_SIZE, random_state=RANDOM_STATE, stratify=y # Stratify ensures similar sentiment distribution in train/test
)
print(f"Training set size: {len(X_train)} samples")
print(f"Test set size: {len(X_test)} samples")

# Initialize TF-IDF Vectorizer
print("\nInitializing TF-IDF Vectorizer...")
vectorizer = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    stop_words='english' # Use built-in English stop words
)

# Fit the vectorizer to the training data and transform it
print("Fitting TF-IDF vectorizer on training data and transforming...")
X_train_tfidf = vectorizer.fit_transform(X_train)

# Transform the test data using the same fitted vectorizer
print("Transforming test data...")
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF transformation complete.")
print(f"Shape of TF-IDF matrix for training data: {X_train_tfidf.shape}")
print(f"Shape of TF-IDF matrix for test data: {X_test_tfidf.shape}")

# --- End Step 3 ---

# (Model Training is next)

Successfully loaded 50000 reviews.

Preprocessing text data...
Preprocessing complete.

Splitting data into training (80%) and testing (20%) sets...
Training set size: 40000 samples
Test set size: 10000 samples

Initializing TF-IDF Vectorizer...
Fitting TF-IDF vectorizer on training data and transforming...
Transforming test data...
TF-IDF transformation complete.
Shape of TF-IDF matrix for training data: (40000, 5000)
Shape of TF-IDF matrix for test data: (10000, 5000)


In [ ]:
import pandas as pd
import os
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression # Import Logistic Regression
from sklearn.metrics import accuracy_score # Import accuracy_score for evaluation

# --- Configuration ---
DATA_FILE = 'movie_data.csv'
TEST_SET_SIZE = 0.2
RANDOM_STATE = 42
TFIDF_MAX_FEATURES = 5000
# --- End Configuration ---

df = None

# ... (Keep data loading code) ...
# Check if the file exists
if not os.path.exists(DATA_FILE):
    print(f"Error: Data file '{DATA_FILE}' not found.")
    exit()
# Load data
try:
    df = pd.read_csv(DATA_FILE)
    print(f"Successfully loaded {len(df)} reviews.")
except Exception as e_csv:
    try:
        df = pd.read_csv(DATA_FILE, compression='gzip')
        print(f"Successfully loaded {len(df)} reviews (as gzipped CSV).")
    except Exception as e_gzip:
        print(f"Failed to load data: {e_gzip}")
        exit()


# --- Step 2: Preprocessing ---
def preprocess_text(text):
    # ... (keep preprocess_text function) ...
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("\nPreprocessing text data...")
df['cleaned_review'] = df['review'].apply(preprocess_text)
print("Preprocessing complete.")
# --- End Step 2 ---


# --- Step 3: Feature Extraction ---
X = df['cleaned_review']
y = df['sentiment']

print(f"\nSplitting data...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SET_SIZE, random_state=RANDOM_STATE, stratify=y
)
# print(f"Training set size: {len(X_train)} samples") # Can comment out
# print(f"Test set size: {len(X_test)} samples")

print("\nInitializing TF-IDF Vectorizer...")
vectorizer = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    stop_words='english'
)

print("Fitting TF-IDF vectorizer on training data and transforming...")
X_train_tfidf = vectorizer.fit_transform(X_train)

print("Transforming test data...")
X_test_tfidf = vectorizer.transform(X_test)
print("TF-IDF transformation complete.")
# print(f"Shape of TF-IDF matrix for training data: {X_train_tfidf.shape}") # Can comment out
# print(f"Shape of TF-IDF matrix for test data: {X_test_tfidf.shape}")
# --- End Step 3 ---


# --- Step 4: Train Model ---

print("\nInitializing Logistic Regression model...")
# Increase max_iter if it doesn't converge with default (100)
model = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)

print("Training the model...")
# Train the model on the TF-IDF transformed training data
model.fit(X_train_tfidf, y_train)
print("Model training complete.")

# Evaluate the model on the test set
print("\nEvaluating the model on the test set...")
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)

print(f"Test Set Accuracy: {accuracy:.4f}") # Print accuracy formatted to 4 decimal places

# --- End Step 4 ---

# (Saving the model is next)

Successfully loaded 50000 reviews.

Preprocessing text data...
Preprocessing complete.

Splitting data...

Initializing TF-IDF Vectorizer...
Fitting TF-IDF vectorizer on training data and transforming...
Transforming test data...
TF-IDF transformation complete.

Initializing Logistic Regression model...
Training the model...
Model training complete.

Evaluating the model on the test set...
Test Set Accuracy: 0.8814


In [ ]:
!pip install joblib

In [ ]:
import pandas as pd
import os
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import joblib

# --- Configuration ---
DATA_FILE = 'movie_data.csv'
TEST_SET_SIZE = 0.2
RANDOM_STATE = 42
TFIDF_MAX_FEATURES = 5000
MODEL_FILENAME = 'sentiment_model.joblib'
VECTORIZER_FILENAME = 'tfidf_vectorizer.joblib'
# --- End Configuration ---

df = None # Initialize df to None

# Check if the file exists FIRST
if not os.path.exists(DATA_FILE):
    print(f"Error: Data file '{DATA_FILE}' not found in the directory '{os.getcwd()}'.")
    exit() # Stop execution if file is missing

# --- Data Loading Block ---
try:
    # Try loading as a regular CSV first (most likely scenario)
    df = pd.read_csv(DATA_FILE)
    print(f"Successfully loaded {len(df)} reviews (as plain CSV).")
except Exception as e_csv:
    print(f"Info: Failed to load as plain CSV ({type(e_csv).__name__}: {e_csv}). Trying gzipped...")
    # If that fails, try loading as gzipped CSV
    try:
        df = pd.read_csv(DATA_FILE, compression='gzip')
        print(f"Successfully loaded {len(df)} reviews (as gzipped CSV).")
    except Exception as e_gzip:
        print(f"Error: Failed to load '{DATA_FILE}' as both plain and gzipped CSV.")
        print(f"Specific error on gzipped attempt: {type(e_gzip).__name__}: {e_gzip}")
        df = None # Explicitly set df to None if both fail
        # exit() # Exit below based on df being None

# --- ADD EXPLICIT CHECK AFTER LOADING ---
if df is None:
    print("CRITICAL ERROR: DataFrame 'df' could not be loaded. Exiting.")
    exit() # Stop script definitively if df wasn't loaded
else:
    # Optional: Check if 'review' column exists
    if 'review' not in df.columns:
        print(f"CRITICAL ERROR: Loaded DataFrame does not contain a 'review' column. Columns found: {df.columns.tolist()}")
        exit()
    print(f"Data loaded successfully. DataFrame shape: {df.shape}")
# --- End Explicit Check ---


# --- Step 2: Preprocessing ---
def preprocess_text(text):
    """Cleans the input text."""
    # Safety check: handle potential non-string data
    if not isinstance(text, str):
        # print(f"Warning: Encountered non-string data in 'review' column: {text}. Replacing with empty string.")
        return "" # Return an empty string or handle as appropriate
    # Convert to lowercase
    text = text.lower()
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove punctuation and numbers (keep only letters and spaces)
    text = re.sub(r'[^a-z\s]', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("\nPreprocessing text data...")
# This line should now be safe because we checked df and 'review' column above
df['cleaned_review'] = df['review'].apply(preprocess_text)
print("Preprocessing complete.")
# --- End Step 2 ---

# --- Step 3: Feature Extraction ---
# Ensure 'cleaned_review' and 'sentiment' columns exist before using them
if 'cleaned_review' not in df.columns or 'sentiment' not in df.columns:
     print("CRITICAL ERROR: Required columns ('cleaned_review', 'sentiment') not found in DataFrame after preprocessing.")
     exit()

X = df['cleaned_review']
y = df['sentiment']

# ...(rest of the script: split, vectorize, train, evaluate, save)...
# ... (No changes needed in the rest of the script from the previous successful version) ...

print(f"\nSplitting data...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SET_SIZE, random_state=RANDOM_STATE, stratify=y
)

print("\nInitializing TF-IDF Vectorizer...")
vectorizer = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    stop_words='english'
)

print("Fitting TF-IDF vectorizer on training data and transforming...")
X_train_tfidf = vectorizer.fit_transform(X_train)

print("Transforming test data...")
X_test_tfidf = vectorizer.transform(X_test)
print("TF-IDF transformation complete.")

# --- Step 4: Train Model ---
print("\nInitializing Logistic Regression model...")
model = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)

print("Training the model...")
model.fit(X_train_tfidf, y_train)
print("Model training complete.")

print("\nEvaluating the model on the test set...")
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Set Accuracy: {accuracy:.4f}")
# --- End Step 4 ---


# --- Step 5: Save Model and Vectorizer ---
print("\nSaving the trained model...")
joblib.dump(model, MODEL_FILENAME)
print(f"Model saved to {MODEL_FILENAME}")

print("\nSaving the TF-IDF vectorizer...")
joblib.dump(vectorizer, VECTORIZER_FILENAME)
print(f"Vectorizer saved to {VECTORIZER_FILENAME}")
# --- End Step 5 ---

print("\nTraining script finished successfully.")

Successfully loaded 50000 reviews (as plain CSV).
Data loaded successfully. DataFrame shape: (50000, 2)

Preprocessing text data...
Preprocessing complete.

Splitting data...

Initializing TF-IDF Vectorizer...
Fitting TF-IDF vectorizer on training data and transforming...
Transforming test data...
TF-IDF transformation complete.

Initializing Logistic Regression model...
Training the model...
Model training complete.

Evaluating the model on the test set...
Test Set Accuracy: 0.8814

Saving the trained model...
Model saved to sentiment_model.joblib

Saving the TF-IDF vectorizer...
Vectorizer saved to tfidf_vectorizer.joblib

Training script finished successfully.


In [ ]:
!pip install Flask

In [ ]:
!pip install flask-ngrok

In [6]:
!pip install pyngrok


In [ ]:
import joblib
import re
from flask import Flask, request, jsonify
import os
# Remove flask_ngrok import
from pyngrok import ngrok # <--- Import pyngrok

# --- Configuration ---
MODEL_FILENAME = 'sentiment_model.joblib'
VECTORIZER_FILENAME = 'tfidf_vectorizer.joblib'
MODEL_PATH = MODEL_FILENAME
VECTORIZER_PATH = VECTORIZER_FILENAME
# --- End Configuration ---

# --- Load Model and Vectorizer ---
# (Loading code remains the same)
print(f"Current Working Directory: {os.getcwd()}")
print("Looking for model and vectorizer files...")
if not os.path.exists(MODEL_PATH): exit(f"Error: Model not found at {os.path.abspath(MODEL_PATH)}")
if not os.path.exists(VECTORIZER_PATH): exit(f"Error: Vectorizer not found at {os.path.abspath(VECTORIZER_PATH)}")
print(f"Found model at: {os.path.abspath(MODEL_PATH)}")
print(f"Found vectorizer at: {os.path.abspath(VECTORIZER_PATH)}")
print("Loading model and vectorizer...")
try:
    model = joblib.load(MODEL_PATH)
    vectorizer = joblib.load(VECTORIZER_PATH)
    print("Model and vectorizer loaded successfully.")
except Exception as e: exit(f"Error loading model/vectorizer: {e}")
# --- End Loading ---

# --- Preprocessing Function ---
# (Preprocessing function remains the same)
def preprocess_text(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
# --- End Preprocessing ---

# --- Create Flask App ---
app = Flask(__name__)
# Remove the run_with_ngrok(app) line
# --- End Flask App ---

# --- Define API Endpoints ---
# (API endpoints remain the same)
@app.route('/', methods=['GET'])
def index():
    return jsonify({"message": "Sentiment Analysis API is running!"})

@app.route('/predict', methods=['POST'])
def predict():
    # ...(predict function remains exactly the same)...
    try:
        if not request.is_json: return jsonify({"error": "Request must be JSON"}), 400
        data = request.get_json()
        if 'text' not in data: return jsonify({"error": "Missing 'text' key"}), 400
        input_text = data['text']
        if not isinstance(input_text, str) or not input_text.strip():
             return jsonify({"error": "'text' must be a non-empty string"}), 400
    except Exception as e:
        return jsonify({"error": f"Invalid JSON format: {e}"}), 400

    processed_text = preprocess_text(input_text)

    try:
        vectorized_text = vectorizer.transform([processed_text])
    except Exception as e:
         print(f"Error during vectorization: {e}")
         return jsonify({"error": "Failed to process input text"}), 500

    try:
        probabilities = model.predict_proba(vectorized_text)[0]
        prediction = model.predict(vectorized_text)[0]
        sentiment_label = 'positive' if prediction == 1 else 'negative'
        confidence_score = probabilities[prediction]
        response = {
            'input_text': input_text,
            'predicted_sentiment': sentiment_label,
            'confidence_score': float(confidence_score)
        }
        return jsonify(response), 200
    except Exception as e:
        print(f"Error during prediction: {e}")
        return jsonify({"error": "Model prediction failed"}), 500
# --- End API Endpoints ---

# --- Run the Flask App ---
if __name__ == '__main__':
    port = 5000 # Define the port Flask will run on internally
    # Open a ngrok tunnel to the Flask app
    public_url = ngrok.connect(port, "http") # Use "http" instead of default "tcp"
    print(f" * ngrok tunnel \"{public_url}\" -> \"http://127.0.0.1:{port}\"")

    # Update any base URLs if necessary (usually not needed for simple APIs)
    # app.config["BASE_URL"] = public_url

    print(f" * Starting Flask app on http://127.0.0.1:{port}")
    # Run Flask app normally on the defined port
    # Turn off Flask's reloader if it causes issues with ngrok (debug=False)
    app.run(port=port, debug=False)
# --- End Run App ---

Current Working Directory: /content
Looking for model and vectorizer files...
Found model at: /content/sentiment_model.joblib
Found vectorizer at: /content/tfidf_vectorizer.joblib
Loading model and vectorizer...
 * ngrok tunnel "NgrokTunnel: "https://b787-34-80-216-212.ngrok-free.app" -> "http://localhost:5000"" -> "http://127.0.0.1:5000"
 * Starting Flask app on http://127.0.0.1:5000
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [30/Apr/2025 09:29:48] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [30/Apr/2025 09:29:49] "GET /favicon.ico HTTP/1.1" 404 -


In [8]:
# Add your ngrok authtoken
!ngrok config add-authtoken 2wPLoSOmTWA0vEEF6ZdGlu6y65a_4rexcJd7vtUX3rCqtQnkp

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
